## Ingest data to Silver with transformations

In [0]:
%py
from pyspark.sql import functions as f

In [0]:
%py
df_dim_categoria = spark.read.table("urban_catalog.bronze.categoria")

In [0]:
%py
df_dim_producto = spark.read.table("urban_catalog.bronze.producto")

In [0]:
%py
df_fact_venta_encabezado = spark.read.table("urban_catalog.bronze.ventaEncabezado")

In [0]:
%py
df_fact_venta_detalles = spark.read.table("urban_catalog.bronze.ventaDetalle")

In [0]:
%py
df_sales = df_fact_venta_encabezado.join(df_fact_venta_detalles, df_fact_venta_encabezado.VentaID == df_fact_venta_detalles.VentaID)\
.select(df_fact_venta_encabezado.VentaID, df_fact_venta_encabezado.FechaVenta, df_fact_venta_encabezado.Canal, df_fact_venta_detalles.ProductoID, df_fact_venta_detalles.Cantidad, 
        df_fact_venta_detalles.PrecioUnitario, df_fact_venta_detalles.Descuento, df_fact_venta_detalles.Subtotal)

In [0]:
%py
df_producto = df_dim_producto.join(df_dim_categoria, df_dim_producto.CategoriaID == df_dim_categoria.CategoriaID)\
    .select(df_dim_producto.ProductoID, df_dim_producto.Nombre.alias("ProductoNombre"), df_dim_categoria.Nombre.alias("Categoria"), df_dim_producto.Genero, df_dim_producto.Talla, df_dim_producto.Color, df_dim_producto.PrecioUnitario)

In [0]:
%py
# Write tables
df_sales.write.mode("overwrite").format("delta").saveAsTable("urban_catalog.silver.sales")


In [0]:
%py
df_producto.write.mode("overwrite").format("delta").saveAsTable("urban_catalog.silver.producto")